# qrange-study variant

Trains the CSD (no-attention) model for a single chosen `(qmin, qmax)` pair. Set `PAIR_INDEX` in the config cell to switch pairs. All outputs are namespaced by the qrange tag so the baseline work is never overwritten.


In [ ]:
import numpy as np
np.random.seed(1337) 
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import MinMaxScaler, StandardScaler, MaxAbsScaler, Normalizer
from keras.models import Sequential
from keras.layers import Dense, Conv1D, BatchNormalization, MaxPooling1D, LeakyReLU, Flatten, Dropout
from keras.callbacks import History, ModelCheckpoint, EarlyStopping
from keras import backend as K
from keras import metrics, regularizers
from keras.optimizers import Adam, Adagrad, Adadelta, RMSprop
from keras_tuner import RandomSearch, Hyperband
from keras_tuner.engine.hyperparameters import HyperParameters
from sklearn.metrics import confusion_matrix, recall_score, f1_score, precision_score
import keras.regularizers
import keras
import glob
import os
import itertools
import math

# Python 3.7 compatibility for keras_tuner (uses math.prod)
if not hasattr(math, "prod"):
    def _prod(values):
        result = 1
        for value in values:
            result *= int(value)
        return result

    math.prod = _prod

In [ ]:
# Import path configuration
import sys
import importlib
sys.path.insert(0, '../..')  # project root (qrange-study/ is one level deeper)

import config
importlib.reload(config)
from config import get_path, qrange_paths, qrange_tag, QRANGE_PAIRS

# --- qrange-study: choose the (qmin, qmax) pair to train on -----------------
PAIR_INDEX = 0  # index into QRANGE_PAIRS in config.py
QMIN, QMAX = QRANGE_PAIRS[PAIR_INDEX]
TAG = qrange_tag(QMIN, QMAX)
_qrange = qrange_paths(QMIN, QMAX)
print(f"qrange-study: training CSD model for (qmin={QMIN}, qmax={QMAX})  [{TAG}]")

csd_pdfs_dir = _qrange['csd_calculated_pdfs']
labels_dir = _qrange['labels']
models_dir = _qrange['models'] / 'csd_no_attention'
results_dir = _qrange['results'] / 'csd'
figures_dir = _qrange['figures']
for _d in (labels_dir, models_dir, results_dir, figures_dir):
    _d.mkdir(parents=True, exist_ok=True)

In [ ]:
# Get calculated PDFs from the namespaced CSD folder for this qrange
print(f"Reading PDFs from: {csd_pdfs_dir}")
files_calc = glob.glob(str(csd_pdfs_dir / '*.dat'))
print(f"Found {len(files_calc)} PDF files")

In [ ]:
from collections import defaultdict
import matplotlib.pyplot as plt

def analyze_filenames(filenames):
    # Count how many filenames start with each digit from 1 to 9, plus 'p' for 10
    counts = defaultdict(int)
    for filepath in filenames:
        filename = os.path.basename(filepath)
        if filename[0].isdigit() and filename[0] != '0':
            counts[filename[0]] += 1
        elif filename[0] == 'p':
            counts['10'] += 1
            
    return counts

counts = analyze_filenames(files_calc)
# Include nuclearity 10 for CSD structures (coordination polymers)
sorted_counts = {str(digit): counts.get(str(digit), 0) for digit in range(1, 10)}
sorted_counts['polymer'] = counts.get('10', 0)

print("Counts of CSD structures by their nuclearity:", sorted_counts)
print(f"Total number of structures: {len(files_calc)}")

digits = list(sorted_counts.keys())
values = list(sorted_counts.values())

plt.figure(figsize=(10, 5))
plt.bar(digits, values, color='steelblue')
plt.xlabel('Nuclearity')
plt.ylabel('Counts')
plt.title('Distribution of CSD structures by nuclearity')
plt.show()

In [ ]:
# (dirs set in the qrange config cell above)
labels_file = labels_dir / f'csd_labels_{TAG}.txt'

raw_data_points = []

with open(labels_file, 'w') as labels_out:
    for f in files_calc:
        df = pd.read_csv(f, usecols=[1], skiprows=201, header=None, delim_whitespace=True, skipfooter=800, engine='python')
        raw_data_points.append(df.values.ravel())
        # Extract just the filename from the full path
        filename = os.path.basename(f)
        if filename[0] == 'p':
            labels_out.write('10')
            labels_out.write('\n')
        else:    
            labels_out.write(filename[0])
            labels_out.write('\n')

raw_data_points = np.array(raw_data_points)

# Load the labels
labels = pd.read_csv(labels_file, header=None)
labels = labels.values.ravel()  # convert the labels to a 1D array

print(f"Labels saved to: {labels_file}")
print(f"Models will be saved to: {models_dir}")

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
normalize = Normalizer()
data_points = normalize.fit_transform(raw_data_points)
fig, ax = plt.subplots()
ax.set_xlim(2,12)
ax.plot(np.arange(len(data_points[77,:]))/100+2, data_points[77,:])
plt.xlabel('r, Å')
plt.ylabel('G(r), $Å^{-2}$')
print(labels[77])

In [ ]:
# Hyperparameter tuning

def build_model(hp):
    model = Sequential()
    model.add(Conv1D(filters=hp.Choice('filters1', [8, 16, 32]),
                     kernel_size=hp.Choice('kernel_size1', [64, 128, 256]),
                     activation='relu', input_shape=(1000, 1)))
    model.add(BatchNormalization())
    model.add(MaxPooling1D(pool_size=2))
    model.add(Dropout(hp.Float('dropout', 0.1, 0.9, step=0.1)))
    model.add(Conv1D(filters=hp.Choice('filters2', [32, 64, 128]),
                     kernel_size=hp.Choice('kernel_size2', [16, 32, 64]),
                     activation='relu'))
    model.add(Dropout(hp.Float('dropout', 0.1, 0.9, step=0.1)))
    model.add(Flatten())
    model.add(Dense(units=hp.Choice('dense_units', [64, 128, 256]), activation='relu', kernel_regularizer=keras.regularizers.l2(0.01)))
    model.add(Dropout(hp.Float('dropout', 0.1, 0.9, step=0.1)))
    model.add(Dense(units=11, activation='softmax', kernel_regularizer=keras.regularizers.l2(0.01)))
    optimizer = Adam(learning_rate=hp.Float('learning_rate', 1e-4, 1e-1, sampling='log'))
    model.compile(optimizer=optimizer, loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(data_points, labels, test_size=0.2, random_state=42, stratify=labels)

In [ ]:

# Setup hyperband directory within models folder
tuner = Hyperband(
    build_model,
    objective='val_accuracy',
    max_epochs=200,
    factor=3,
    directory=str(models_dir),
    project_name=f'hyperband_csd_2-12_{TAG}',
    overwrite = True
)

tuner.search(X_train, y_train, epochs=200, validation_data=(X_val, y_val), callbacks=[EarlyStopping(monitor='val_loss', patience=3)])
tuner.search_space_summary()
best_hyperparameters = tuner.get_best_hyperparameters(num_trials=1)[0]
print("Best hyperparameters:")
print(best_hyperparameters.values)

# Extract best hyperparameters for retraining
best_filters1 = best_hyperparameters.get('filters1')
best_kernel_size1 = best_hyperparameters.get('kernel_size1')
best_filters2 = best_hyperparameters.get('filters2')
best_kernel_size2 = best_hyperparameters.get('kernel_size2')
best_dropout = best_hyperparameters.get('dropout')
best_dense_units = best_hyperparameters.get('dense_units')
best_learning_rate = best_hyperparameters.get('learning_rate')

print(f"\nExtracted hyperparameters for retraining:")
print(f"  filters1: {best_filters1}, kernel_size1: {best_kernel_size1}")
print(f"  filters2: {best_filters2}, kernel_size2: {best_kernel_size2}")
print(f"  dropout: {best_dropout}, dense_units: {best_dense_units}")
print(f"  learning_rate: {best_learning_rate}")

best_model = tuner.get_best_models(num_models=1)[0]

# Save best model to models directory
best_model_path = models_dir / f'tuned_csd_2-12_{TAG}.h5'
best_model.save(best_model_path)
print(f"\nBest model saved to: {best_model_path}")

In [ ]:
def create_model():
    # Create model using best hyperparameters from tuning
    model = Sequential()
    # Add the convolutional layers
    model.add(Conv1D(filters=best_filters1, kernel_size=best_kernel_size1, activation='relu', input_shape=(1000, 1)))
    model.add(BatchNormalization())
    model.add(MaxPooling1D(pool_size=2))
    model.add(Dropout(best_dropout))
    model.add(Conv1D(filters=best_filters2, kernel_size=best_kernel_size2, activation='relu'))
    model.add(Dropout(best_dropout))

    # Flatten the output of the convolutional layers
    model.add(Flatten())

    # Add the fully connected layers
    model.add(Dense(units=best_dense_units, activation='relu', kernel_regularizer=keras.regularizers.l2(0.01)))
    model.add(Dropout(best_dropout))
    model.add(Dense(units=11, activation='softmax', kernel_regularizer=keras.regularizers.l2(0.01)))

    # Compile the model with the best learning rate
    optimizer = Adam(learning_rate=best_learning_rate)
    model.compile(optimizer=optimizer, loss='sparse_categorical_crossentropy', metrics=['accuracy'])

    return model

In [ ]:
num_folds = 5
num_epochs = 200
kf = StratifiedKFold(n_splits=num_folds, shuffle=True, random_state=42)
fold_num = 1 
all_fold_results = []
histories = []

In [ ]:
# Setup directory for fold models
folds_dir = models_dir / 'cv_folds'
folds_dir.mkdir(parents=True, exist_ok=True)

for train_index, val_index in kf.split(data_points, labels):
    X_train, X_val = data_points[train_index], data_points[val_index]
    y_train, y_val = labels[train_index], labels[val_index]

    model = create_model()

    checkpoint_path = folds_dir / f"csd_fold_{fold_num}_model_{TAG}.h5"
    checkpoint = ModelCheckpoint(str(checkpoint_path), monitor='val_accuracy', mode='max', verbose=1, save_best_only=True)

    print(f'Training fold {fold_num}...')
    history = model.fit(X_train, y_train, epochs=num_epochs, validation_data=(X_val, y_val), callbacks=[checkpoint])
    histories.append(history)

    # Load the best model saved by the checkpoint and evaluate
    load_model = keras.models.load_model(str(checkpoint_path))
    
    val_loss, val_acc = load_model.evaluate(X_val, y_val)
    print(f'Fold {fold_num} accuracy:', val_acc)

    all_fold_results.append(val_acc)

    fold_num += 1

# Print overall performance across all folds
print('All fold accuracies:', all_fold_results)
print('Mean accuracy:', np.mean(all_fold_results))
print('Standard deviation:', np.std(all_fold_results))
print(f"\nFold models saved to: {folds_dir}")

In [ ]:
# Identifying the best fold (assuming higher validation accuracy is better)
best_fold_index = np.argmax([np.max(hist.history['val_accuracy']) for hist in histories])

# Collect metrics across all folds (truncate to shortest fold length)
min_epochs = min(len(h.history['accuracy']) for h in histories)
all_train_acc = np.array([h.history['accuracy'][:min_epochs] for h in histories])
all_val_acc = np.array([h.history['val_accuracy'][:min_epochs] for h in histories])
all_train_loss = np.array([h.history['loss'][:min_epochs] for h in histories])
all_val_loss = np.array([h.history['val_loss'][:min_epochs] for h in histories])

epochs = np.arange(min_epochs)

plt.figure(figsize=(12, 5))

# Accuracy plot
plt.subplot(1, 2, 1)
plt.fill_between(epochs, all_train_acc.min(axis=0), all_train_acc.max(axis=0), alpha=0.2, color='tab:blue')
plt.plot(epochs, all_train_acc.mean(axis=0), color='tab:blue', linewidth=1.5, label='Train Accuracy')
plt.fill_between(epochs, all_val_acc.min(axis=0), all_val_acc.max(axis=0), alpha=0.2, color='tab:orange')
plt.plot(epochs, all_val_acc.mean(axis=0), color='tab:orange', linewidth=1.5, label='Validation Accuracy')
plt.title('Accuracy across all folds')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()

# Loss plot
plt.subplot(1, 2, 2)
plt.fill_between(epochs, all_train_loss.min(axis=0), all_train_loss.max(axis=0), alpha=0.2, color='tab:blue')
plt.plot(epochs, all_train_loss.mean(axis=0), color='tab:blue', linewidth=1.5, label='Train Loss')
plt.fill_between(epochs, all_val_loss.min(axis=0), all_val_loss.max(axis=0), alpha=0.2, color='tab:orange')
plt.plot(epochs, all_val_loss.mean(axis=0), color='tab:orange', linewidth=1.5, label='Validation Loss')
plt.title('Loss across all folds')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()

plt.show()

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(data_points, labels, test_size=0.2, random_state=42)

# Load the best model from cross-validation (best fold determined from training)
best_fold = np.argmax(all_fold_results) + 1  # +1 because fold_num starts at 1
best_model_path = models_dir / 'cv_folds' / f'csd_fold_{best_fold}_model_{TAG}.h5'
print(f"Loading best model from fold {best_fold}: {best_model_path}")

load_model = keras.models.load_model(best_model_path)

load_model.evaluate(X_val, y_val)
y_pred_prob = load_model.predict(X_val)
y_pred = np.argmax(y_pred_prob, axis=1)
confusion = confusion_matrix(y_val, y_pred)
recall = recall_score(y_val, y_pred, average=None)
f1 = f1_score(y_val, y_pred, average=None)
precision = precision_score(y_val, y_pred, average=None)
#print('Confusion matrix', confusion)
print('Recall score:', recall)
print('F1 score:', f1)
print('Precision score:', precision)

import matplotlib.pyplot as plt

def plot_confusion_matrix(cm, classes,
                          normalize=False,
                          title='Confusion matrix',
                          cmap=plt.cm.Blues):
    """
    This function prints and plots the confusion matrix.
    Normalization can be applied by setting `normalize=True`.
    """
    if normalize:
        cm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
        print("Normalized confusion matrix")
    else:
        print('Confusion matrix, without normalization')

    print(cm)

    plt.imshow(cm, interpolation='nearest', cmap=cmap)
    plt.title(title)
    plt.colorbar()
    tick_marks = np.arange(len(classes))
    plt.xticks(tick_marks, classes, rotation=45)
    plt.yticks(tick_marks, classes)

    fmt = '.2f' if normalize else 'd'
    thresh = cm.max() / 2.
    for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
        plt.text(j, i, format(cm[i, j], fmt),
                 horizontalalignment="center",
                 color="white" if cm[i, j] > thresh else "black")

    plt.tight_layout()
    plt.ylabel('True label')
    plt.xlabel('Predicted label')

# Plot the confusion matrix
plot_confusion_matrix(confusion, classes = np.unique(y_val), title='Confusion matrix', normalize=False)
plt.show()

In [ ]:
# Attention layer extraction skipped - no attention mechanism in this model version

In [ ]:
# Skipping attention weights plot - no attention mechanism in this model version
print("Note: This model does not include an attention mechanism, so attention weight visualization is not applicable.")

In [ ]:
import tensorflow as tf
import matplotlib.pyplot as plt

# Use the loaded model from cross-validation
model = load_model

# Compile the model
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

sample_index = 6  
sample_label = y_train[sample_index]
some_input = tf.convert_to_tensor(X_train[sample_index].reshape(1, 1000, 1), dtype=tf.float32)

with tf.GradientTape() as tape:
    tape.watch(some_input)
    prediction = model(some_input)
    loss = prediction[0][sample_label]

grad_values = tape.gradient(loss, some_input)
grad_numpy = grad_values.numpy().reshape(1000)
print(f"Sample label: {y_train[sample_index]}")
angstrom_range = 2 + np.linspace(0, 1000, 1000) / 100

# Plot
plt.figure(figsize=(12, 3.3))
plt.plot(angstrom_range, grad_numpy)
plt.title('Saliency Map')
plt.xlabel('r, Å')
plt.ylabel('Gradient Value')
plt.grid(True)
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

sample_size = 400
fig, axes = plt.subplots(4, 3, figsize=(15, 20))

# Loop through the unique labels (from 1 to 10)
for label in range(1, 11):
    ax = axes[(label - 1) // 3, (label - 1) % 3]
    gradients = []
    
    # Loop through first `sample_size` samples to check for this label
    for idx in range(min(sample_size, len(y_train))):
        if y_train[idx] == label:
            sample_input = tf.convert_to_tensor(X_train[idx].reshape(1, 1000, 1), dtype=tf.float32)
            
            with tf.GradientTape() as tape:
                tape.watch(sample_input)
                prediction = model(sample_input)
                loss = prediction[0][label - 1]  # Adjust for 0-based index
            
            grad_values = tape.gradient(loss, sample_input)
            grad_numpy = grad_values.numpy().reshape(1000)
            gradients.append(grad_numpy)
            
            angstrom_range = 2 + np.linspace(0, 1000, 1000) / 100
            ax.plot(angstrom_range, grad_numpy, alpha=0.2, color='royalblue')
    
    # Plot the average line if we have any gradients for this label
    if gradients:
        mean_gradient = np.mean(gradients, axis=0)
        ax.plot(angstrom_range, mean_gradient, color='red')
    
    ax.set_title(f'Label {label}')
    ax.set_xlabel('r, Å')
    ax.set_ylabel('Gradient Value')
    ax.grid(True)

fig.delaxes(axes[3,1])
fig.delaxes(axes[3,2])

plt.tight_layout()

# figures_dir set in the qrange config cell above
plt.savefig(figures_dir / f'csd_saliency_maps_no_attention_{TAG}.png', dpi=400)
plt.show()